# 03 - Data Preparation

This notebook converts the raw minute-level electricity readings into a clean hourly dataset for analysis, modeling, and dashboard use.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "DataSet" / "household_power_consumption.csv"
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "prepared_hourly_energy.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## Load and Standardize Columns

Column names are converted to lowercase snake case so later cells are easier to read.

In [2]:
raw_df = pd.read_csv(DATA_PATH, na_values=["?"], low_memory=False)
raw_df.columns = [column.strip().lower().replace(" ", "_") for column in raw_df.columns]
display(raw_df.head())

,date,time,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Convert Date, Time, and Numeric Fields

The original file stores date and time separately. They are combined into a single `datetime` column, and power measurements are converted into numeric types.

In [3]:
numeric_columns = [
    "global_active_power",
    "global_reactive_power",
    "voltage",
    "global_intensity",
    "sub_metering_1",
    "sub_metering_2",
    "sub_metering_3",
]

clean_df = raw_df.copy()
clean_df["datetime"] = pd.to_datetime(
    clean_df["date"].astype(str) + " " + clean_df["time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

clean_df = clean_df.dropna(subset=["datetime"]).sort_values("datetime")
clean_df = clean_df.drop_duplicates(subset=["datetime"]).reset_index(drop=True)
print("Rows after datetime cleaning:", len(clean_df))

Rows after datetime cleaning: 1048575


## Handle Missing Values

Because the readings are a time series, missing numeric values are interpolated over time. Remaining edge gaps are filled forward and backward.

In [4]:
clean_df = clean_df.set_index("datetime")
clean_df[numeric_columns] = (
    clean_df[numeric_columns]
    .interpolate(method="time", limit_direction="both")
    .ffill()
    .bfill()
)
clean_df = clean_df.reset_index()
print("Missing values after interpolation:", clean_df[numeric_columns].isna().sum().sum())

Missing values after interpolation: 0


## Feature Engineering

The new features support time analysis, appliance-level interpretation, classification, and forecasting.

In [5]:
clean_df["hour"] = clean_df["datetime"].dt.hour
clean_df["day_of_week"] = clean_df["datetime"].dt.dayofweek
clean_df["month"] = clean_df["datetime"].dt.month
clean_df["year"] = clean_df["datetime"].dt.year
clean_df["is_weekend"] = clean_df["day_of_week"].isin([5, 6]).astype(int)

clean_df["sub_metering_total_wh"] = clean_df[["sub_metering_1", "sub_metering_2", "sub_metering_3"]].sum(axis=1)
clean_df["active_energy_wh"] = clean_df["global_active_power"] * 1000 / 60
clean_df["unmetered_energy_wh"] = (clean_df["active_energy_wh"] - clean_df["sub_metering_total_wh"]).clip(lower=0)

display(clean_df.head())

,datetime,date,time,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3,hour,day_of_week,month,year,is_weekend,sub_metering_total_wh,active_energy_wh,unmetered_energy_wh
0,2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,17,5,12,2006,1,18.0,70.266667,52.266667
1,2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,17,5,12,2006,1,17.0,89.333333,72.333333
2,2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,17,5,12,2006,1,19.0,89.566667,70.566667
3,2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,17,5,12,2006,1,18.0,89.800000,71.800000
4,2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,17,5,12,2006,1,18.0,61.100000,43.100000


## Resample to Hourly Data

Hourly data is smaller, easier to visualize, and more suitable for the baseline models used in this project.

In [6]:
hourly_df = clean_df.set_index("datetime").resample("h").agg({
    "global_active_power": "mean",
    "global_reactive_power": "mean",
    "voltage": "mean",
    "global_intensity": "mean",
    "sub_metering_1": "sum",
    "sub_metering_2": "sum",
    "sub_metering_3": "sum",
    "sub_metering_total_wh": "sum",
    "active_energy_wh": "sum",
    "unmetered_energy_wh": "sum",
}).dropna().reset_index()

hourly_df["hour"] = hourly_df["datetime"].dt.hour
hourly_df["day_of_week"] = hourly_df["datetime"].dt.dayofweek
hourly_df["month"] = hourly_df["datetime"].dt.month
hourly_df["year"] = hourly_df["datetime"].dt.year
hourly_df["is_weekend"] = hourly_df["day_of_week"].isin([5, 6]).astype(int)

threshold = hourly_df["global_active_power"].quantile(0.75)
hourly_df["high_consumption"] = (hourly_df["global_active_power"] >= threshold).astype(int)

hourly_df["lag_1_power"] = hourly_df["global_active_power"].shift(1)
hourly_df["lag_2_power"] = hourly_df["global_active_power"].shift(2)
hourly_df["lag_24_power"] = hourly_df["global_active_power"].shift(24)
hourly_df["rolling_3_power_mean"] = hourly_df["global_active_power"].shift(1).rolling(3, min_periods=1).mean()
hourly_df["rolling_24_power_mean"] = hourly_df["global_active_power"].shift(1).rolling(24, min_periods=1).mean()
hourly_df = hourly_df.bfill().ffill()

print("Hourly shape:", hourly_df.shape)
print("Start:", hourly_df["datetime"].min())
print("End:", hourly_df["datetime"].max())

Hourly shape: (17477, 22)
Start: 2006-12-16 17:00:00
End: 2008-12-13 21:00:00


## Save Prepared Dataset

The prepared file is used by the modeling, evaluation, and dashboard notebooks.

In [7]:
hourly_df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
display(hourly_df.head())

Saved: d:\dm-final-energy-omar\outputs\prepared_hourly_energy.csv


,datetime,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3,sub_metering_total_wh,active_energy_wh,...,day_of_week,month,year,is_weekend,high_consumption,lag_1_power,lag_2_power,lag_24_power,rolling_3_power_mean,rolling_24_power_mean
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,19.0,607.0,626.0,2533.733333,...,5,12,2006,1,1,4.222889,4.222889,4.222889,4.222889,4.222889
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0,1415.0,3632.200000,...,5,12,2006,1,1,4.222889,4.222889,4.222889,4.222889,4.222889
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0,1087.0,3400.233333,...,5,12,2006,1,1,3.632200,4.222889,4.222889,3.927544,3.927544
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0,1007.0,3268.566667,...,5,12,2006,1,1,3.400233,3.632200,4.222889,3.751774,3.751774
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0,1058.0,3056.466667,...,5,12,2006,1,1,3.268567,3.400233,4.222889,3.433667,3.630972
